**General Description**

The following notebook contains the code to create, train, validate, and test a rainfall-runoff model using an LSTM network architecture. The notebook support running experiments in different large-sample hydrology datasets including: CAMELS-GB, CAMELS-US, CAMELS-DE. The details for each dataset can be read from a .yml file.

***Authors:***
- Eduardo Acuña Espinoza (eduardo.espinoza@kit.edu)
- Manuel Alvarez Chaves (manuel.alvarez-chaves@simtech.uni-stuttgart.de)

In [1]:
# Import necessary packages
import datetime
import pickle
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

sys.path.append("..")
# Import classes and functions from other files
from hy2dl.datasetzoo import get_dataset
from hy2dl.evaluation.metrics import nse
from hy2dl.modelzoo import get_model
from hy2dl.training.loss import nse_basin_averaged
from hy2dl.utils.config import Config
from hy2dl.utils.optimizer import Optimizer
from hy2dl.utils.utils import set_random_seed, upload_to_device

# colorblind friendly palette
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}

Part 1. Initialize information

In [2]:
# Create a dictionary where all the information will be stored
experiment_settings = {}

# Experiment name
# experiment_settings["experiment_name"] = "bs_256_uniqueBlocksTrue_random_0.8"
experiment_settings["experiment_name"] = "test"

# paths to access the information
experiment_settings["path_data"] = "../data/CAMELS_DE"
# experiment_settings["path_entities"] = "../data/basin_id/basins_camels_de_hourly_292_Bayern.txt"
experiment_settings["path_entities"] = "../data/basin_id/basins_camels_de_hourly_3.txt"

experiment_settings["dynamic_input"] = [
    "precipitation_mean",
    "precipitation_stdev",
    "radiation_global_mean",
    "temperature_min",
    "temperature_max",
]

experiment_settings["target"] = ["discharge_spec_obs"]

# static attributes that will be used. If one is not using static_inputs, initialize the variable as an empty list.
experiment_settings["static_input"] = [
    "area",
    "elev_mean",
    "clay_0_30cm_mean",
    "sand_0_30cm_mean",
    "silt_0_30cm_mean",
    "artificial_surfaces_perc",
    "agricultural_areas_perc",
    "forests_and_seminatural_areas_perc",
    "wetlands_perc",
    "water_bodies_perc",
    "p_mean",
    "p_seasonality",
    "frac_snow",
    "high_prec_freq",
    "low_prec_freq",
    "high_prec_dur",
    "low_prec_dur",
]

# # # time periods 
# experiment_settings["training_period"] = ["1970-10-01", "1999-12-31"]
# experiment_settings["validation_period"] = ["1965-10-01", "1970-09-30"]
# experiment_settings["testing_period"] = ["2000-01-01", "2020-12-31"]

# # # time periods (for short test)
# experiment_settings["training_period"] = ["1970-10-01", "1971-12-31"]
experiment_settings["training_period"] = ["1970-10-01", "1970-12-31"]
experiment_settings["validation_period"] = ["1965-10-01", "1966-09-30"]
experiment_settings["testing_period"] = ["2000-01-01", "2001-12-31"]

# model configuration
experiment_settings["hidden_size"] = 128
experiment_settings["batch_size_training"] = 256    # original: 256
experiment_settings["batch_size_evaluation"] = 1024
experiment_settings["epochs"] = 1  # 12
experiment_settings["dropout_rate"] = 0.4
experiment_settings["learning_rate"] = 0.001
experiment_settings["steplr_step_size"] = 10
experiment_settings["steplr_gamma"] = 0.5
experiment_settings["validate_every"] = 4
experiment_settings["validate_n_random_basins"] = -1

experiment_settings["seq_length_hindcast"] = 365

# experiment_settings["predict_last_n"] = 24

# experiment_settings["dynamic_embedding"] = {"hiddens": [10, 10, 10]}  # for my case, I do not need it actually
# experiment_settings["custom_seq_processing_flag"] = True

# device to train the model
# experiment_settings["device"] = "cuda:0"
experiment_settings["device"] = "cpu"
experiment_settings["num_workers"] = 0  # ori: 4

# define random seed
experiment_settings["random_seed"] = 110

# dataset
experiment_settings["dataset"] = "camels_de"
# model
experiment_settings["model"] = "CudaLSTM"
experiment_settings["initial_forget_bias"] = 3.0

In [3]:
# Read experiment settings
config = Config(experiment_settings)
config.init_experiment()
config.dump()

Part 2. Create datasets and dataloaders used to train/validate the model

In [4]:
# Get dataset class
Dataset = get_dataset(config)

# Dataset training
config.logger.info(f"Loading training data from {config.dataset} dataset")
total_time = time.time()

training_dataset = Dataset(cfg=config, time_period="training")

training_dataset.calculate_basin_std()
training_dataset.calculate_global_statistics(save_scaler=True)
training_dataset.standardize_data()

config.logger.info(f"Number of entities with valid samples: {len(training_dataset.df_ts)}")
config.logger.info(
    f"Time required to process {len(training_dataset.df_ts)} entities: "
    f"{datetime.timedelta(seconds=int(time.time() - total_time))}"
)
config.logger.info(f"Number of valid training samples: {len(training_dataset)}\n")

# Dataloader training
train_loader = DataLoader(
    dataset=training_dataset,
    batch_size=config.batch_size_training,
    shuffle=True,
    drop_last=True,
    collate_fn=training_dataset.collate_fn,
    num_workers=config.num_workers,
)

# Print details of a loader´s sample to check that the format is correct
config.logger.info("Details training dataloader".center(60, "-"))
config.logger.info(f"Batch structure (number of batches: {len(train_loader)})")
config.logger.info(f"{'Key':^30}|{'Shape':^30}")
# config.logger.info("-" * 60)
# Loop through the sample dictionary and print the shape of each element
for key, value in next(iter(train_loader)).items():
    if key.startswith(("x_d", "x_conceptual")):
        config.logger.info(f"{key}")
        for i, v in value.items():
            config.logger.info(f"{i:^30}|{str(v.shape):^30}")
    else:
        config.logger.info(f"{key:<30}|{str(value.shape):^30}")

config.logger.info("")  # prints a blank line

2025-11-26 21:09:11 - Loading training data from camels_de dataset


Processing entities: 100%|##########| 3/3 [00:00<00:00, 24.87entity/s]

2025-11-26 21:09:11 - Number of entities with valid samples: 3
2025-11-26 21:09:11 - Time required to process 3 entities: 0:00:00
2025-11-26 21:09:11 - Number of valid training samples: 276

2025-11-26 21:09:11 - ----------------Details training dataloader-----------------
2025-11-26 21:09:11 - Batch structure (number of batches: 1)
2025-11-26 21:09:11 -              Key              |            Shape             
2025-11-26 21:09:11 - x_d
2025-11-26 21:09:11 -       precipitation_mean      |    torch.Size([256, 365])    
2025-11-26 21:09:11 -      precipitation_stdev      |    torch.Size([256, 365])    
2025-11-26 21:09:11 -     radiation_global_mean     |    torch.Size([256, 365])    
2025-11-26 21:09:11 -        temperature_min        |    torch.Size([256, 365])    
2025-11-26 21:09:11 -        temperature_max        |    torch.Size([256, 365])    
2025-11-26 21:09:11 - x_s                           |    torch.Size([256, 17])     
2025-11-26 21:09:11 - y_obs                        


D:\Research\Projects\Hy2DL\src\hy2dl\datasetzoo\basedataset.py:395: UserWarning: The standard deviation of wetlands_perc is NaN or zero. The std has been forced to 1 to avoid NaN issues during normalization.
  self.scaler["x_s_std"] = torch.tensor(list(self._check_std(x_s_std).values()), dtype=torch.float32)
D:\Research\Projects\Hy2DL\src\hy2dl\datasetzoo\basedataset.py:395: UserWarning: The standard deviation of water_bodies_perc is NaN or zero. The std has been forced to 1 to avoid NaN issues during normalization.
  self.scaler["x_s_std"] = torch.tensor(list(self._check_std(x_s_std).values()), dtype=torch.float32)
D:\Research\Projects\Hy2DL\src\hy2dl\datasetzoo\basedataset.py:395: UserWarning: The standard deviation of high_prec_dur is NaN or zero. The std has been forced to 1 to avoid NaN issues during normalization.
  self.scaler["x_s_std"] = torch.tensor(list(self._check_std(x_s_std).values()), dtype=torch.float32)
D:\Research\Projects\Hy2DL\src\hy2dl\datasetzoo\basedataset.py:39

In [5]:
# Check the one train sample
dataset_sample = training_dataset[0]
print(f"One sample in training dataset look like: {dataset_sample}")

One sample in training dataset look like: {'x_d': {'precipitation_mean': tensor([-5.7687e-01, -5.8059e-01, -5.8059e-01, -5.8059e-01, -5.8059e-01,
        -5.8059e-01, -5.8059e-01, -5.8059e-01, -5.8059e-01, -5.8059e-01,
        -5.8059e-01, -5.8059e-01, -5.8059e-01, -5.8059e-01, -5.8059e-01,
        -5.8059e-01, -5.8059e-01, -5.8059e-01, -5.8059e-01, -5.8059e-01,
        -5.8059e-01,  3.6451e-01,  1.0689e+00, -4.4212e-01, -5.8059e-01,
        -5.8059e-01, -5.8059e-01, -5.8059e-01, -4.8952e-01, -5.8059e-01,
        -5.8059e-01, -5.8059e-01, -5.8059e-01, -4.6241e-02,  9.0366e-02,
        -5.7966e-01, -4.5420e-01, -2.5347e-01,  8.1615e-01, -5.6851e-01,
        -5.7501e-01,  1.2659e+00,  1.1581e+00,  1.2661e-01, -5.8059e-01,
        -5.8059e-01,  2.5113e-01, -4.1238e-01, -5.8059e-01, -5.8059e-01,
        -5.8059e-01, -5.8059e-01, -5.8059e-01, -6.4827e-02,  6.1914e-01,
         8.6912e-01, -4.6350e-01,  8.7562e-01, -4.6071e-01, -5.4063e-01,
        -5.8059e-01, -5.8059e-01,  8.1801e-01,  3.7

# Generate memmap.npy

In [6]:
len(training_dataset)

276

In [7]:
import numpy as np
import torch
import os

# Choose mode: "pre_train" or "fine-tune"
# mode = "pre_train"
mode = "fine-tune"

# 使用绝对路径避免 Windows 的 memmap bug
memmap_path = "./memmap.npy"
meta_path = "./memmap_meta.npz"

# ===== 创建 memmap =====
data_mm = np.memmap(
    memmap_path,
    dtype="float32",
    mode="w+",
    shape=((len(training_dataset) * config.seq_length_hindcast),
           (len(config.dynamic_input) + len(config.static_input)))
)

start_list = []
length_list = []
file_idx_list = []
label_list = []

row_index = 0

for i in range(len(training_dataset)):
    sample = training_dataset[i]

    # ===== x_d 动态变量 =====
    x_d_dict = sample["x_d"]
    dyn_keys = sorted(x_d_dict.keys())
    dyn_vars = [x_d_dict[k].numpy() for k in dyn_keys]
    x_d = np.stack(dyn_vars, axis=-1)  

    # ===== x_s 静态变量 =====
    x_s = sample["x_s"].numpy()
    x_s = np.tile(x_s, (config.seq_length_hindcast, 1))       

    # ===== 拼接 =====
    merged = np.concatenate([x_d, x_s], axis=-1) 

    # ===== 写入 memmap =====
    data_mm[row_index : row_index + config.seq_length_hindcast] = merged

    # ===== meta =====
    start_list.append(row_index)
    length_list.append(config.seq_length_hindcast)
    file_idx_list.append(0)

    # ===== 记录 label（y_obs） =====
    y_obs = sample["y_obs"].numpy()     
    label_list.append(y_obs)

    row_index += config.seq_length_hindcast

data_mm.flush()

# ===== Save meta =====
if mode=="pre-train":
    np.savez_compressed(
        meta_path,
        start=np.array(start_list),
        length=np.array(length_list),
        shape=np.array([[(len(training_dataset) * config.seq_length_hindcast), (len(config.dynamic_input) + len(config.static_input))]]),
        file_idx=np.array(file_idx_list),
        dtype=np.array("float32"),
        filenames=np.array([memmap_path])
    )
elif mode=="fine-tune":
    np.savez_compressed(
        meta_path,
        start=np.array(start_list),
        length=np.array(length_list),
        shape=np.array([[(len(training_dataset) * config.seq_length_hindcast), (len(config.dynamic_input) + len(config.static_input))]]),
        file_idx=np.array(file_idx_list),
        dtype=np.array("float32"),
        filenames=np.array([memmap_path]),
        label=np.array(label_list, dtype=object)
        )

In [8]:
# 1. 加载元数据
meta = np.load(meta_path, allow_pickle=True)
start = meta["start"]
length = meta["length"]
shape = meta["shape"]
file_idx = meta["file_idx"]
dtype = np.dtype(str(meta["dtype"]))
print("dataset shape: ", shape)  # the first element is total length (sequence lenghth * num_samples), the second is feature dimension (12 for ECG)

# 2. 打开 memmap 文件
memmap_data = np.memmap(memmap_path, dtype=dtype, mode='r', shape=tuple(shape[0]))

# # 3. 访问第一个样本（示例）
idx = 0
sample_start = start[idx]
sample_length = length[idx]

sample = memmap_data[sample_start : sample_start + sample_length]
print("One sample shape:", sample.shape)
print("The first 5 timesteps of this sample: ", sample[:5, :])  # 显示前5个时间步的12维数据

dataset shape:  [[100740     22]]
One sample shape: (365, 22)
The first 5 timesteps of this sample:  [[-0.57687056 -0.519409   -0.3900441   0.08515149  0.3521848  -0.6418174
   0.30368382 -0.9832141   0.8735651  -0.7678422  -0.57735026 -0.6469754
   0.640841    0.          0.         -0.89504695  1.072222   -0.09491571
  -0.07193203 -0.14177522  0.          0.        ]
 [-0.58058774 -0.5988011   0.13945258  0.64386445  0.6758459  -0.6418174
   0.30368382 -0.9832141   0.8735651  -0.7678422  -0.57735026 -0.6469754
   0.640841    0.          0.         -0.89504695  1.072222   -0.09491571
  -0.07193203 -0.14177522  0.          0.        ]
 [-0.58058774 -0.5988011   0.88780606  0.786895    0.38814718 -0.6418174
   0.30368382 -0.9832141   0.8735651  -0.7678422  -0.57735026 -0.6469754
   0.640841    0.          0.         -0.89504695  1.072222   -0.09491571
  -0.07193203 -0.14177522  0.          0.        ]
 [-0.58058774 -0.5988011   0.94140106  0.98803157  0.4334331  -0.6418174
   0.30368382

In [9]:
print("==== Keys in meta file ====")
print(meta.files)

print("\n==== Content ====")
for key in meta.files:
    print(f"\n--- {key} ---")
    print(meta[key])

==== Keys in meta file ====
['start', 'length', 'shape', 'file_idx', 'dtype', 'filenames', 'label']

==== Content ====

--- start ---
[     0    365    730   1095   1460   1825   2190   2555   2920   3285
   3650   4015   4380   4745   5110   5475   5840   6205   6570   6935
   7300   7665   8030   8395   8760   9125   9490   9855  10220  10585
  10950  11315  11680  12045  12410  12775  13140  13505  13870  14235
  14600  14965  15330  15695  16060  16425  16790  17155  17520  17885
  18250  18615  18980  19345  19710  20075  20440  20805  21170  21535
  21900  22265  22630  22995  23360  23725  24090  24455  24820  25185
  25550  25915  26280  26645  27010  27375  27740  28105  28470  28835
  29200  29565  29930  30295  30660  31025  31390  31755  32120  32485
  32850  33215  33580  33945  34310  34675  35040  35405  35770  36135
  36500  36865  37230  37595  37960  38325  38690  39055  39420  39785
  40150  40515  40880  41245  41610  41975  42340  42705  43070  43435
  43800  44165

# To do

In [ ]:
# In evaluation (validation and testing) we will create an individual dataset per basin
config.logger.info(f"Loading validation data from {config.dataset} dataset")
entities_ids = np.loadtxt(config.path_entities_validation, dtype="str").tolist()
iterator = tqdm(
    [entities_ids] if isinstance(entities_ids, str) else entities_ids,
    desc="Processing entities",
    unit="entity",
    ascii=True,
)

total_time = time.time()
validation_dataset = {}
for entity in iterator:
    dataset = Dataset(cfg=config, time_period="validation", check_NaN=False, entities_ids=entity)

    dataset.scaler = training_dataset.scaler
    dataset.standardize_data(standardize_output=False)
    validation_dataset[entity] = dataset

config.logger.info(
    f"Time required to process {len(iterator)} entities: {datetime.timedelta(seconds=int(time.time() - total_time))}\n"
)

Part 3. Train model

In [ ]:
# Initialize model
set_random_seed(cfg=config)
model = get_model(config).to(config.device)

# Initialize optimizer
optimizer = Optimizer(cfg=config, model=model)

# Training report structure
config.logger.info("Training model".center(60, "-"))
config.logger.info(f"{'':^16}|{'Trainining':^21}|{'Validation':^21}|")
config.logger.info(f"{'Epoch':^5}|{'LR':^10}|{'Loss':^10}|{'Time':^10}|{'Metric':^10}|{'Time':^10}|")

total_time = time.time()
# Loop through epochs
for epoch in range(1, config.epochs + 1):
    train_time = time.time()
    loss_evol = []
    # Training -------------------------------------------------------------------------------------------------------
    model.train()
    # Loop through the different batches in the training dataset
    iterator = tqdm(
        train_loader, desc=f"Epoch {epoch}/{config.epochs}. Training", unit="batches", ascii=True, leave=False
    )

    for idx, sample in enumerate(iterator):
        # reach maximum iterations per epoch
        if config.max_updates_per_epoch is not None and idx >= config.max_updates_per_epoch:
            break

        sample = upload_to_device(sample, config.device)  # upload tensors to device
        optimizer.optimizer.zero_grad()  # sets gradients to zero

        # Forward pass of the model
        pred = model(sample)
        # Calcuate loss
        loss = nse_basin_averaged(y_sim=pred["y_hat"], y_obs=sample["y_obs"], per_basin_target_std=sample["std_basin"])

        # Backpropagation (calculate gradients)
        loss.backward()

        # Update model parameters (e.g, weights and biases)
        optimizer.clip_grad_and_step(epoch, idx)

        # Keep track of the loss per batch
        loss_evol.append(loss.item())
        iterator.set_postfix({"loss": f"{np.mean(loss_evol):.3f}"})

        # remove elements from cuda to free memory
        del sample, pred
        torch.cuda.empty_cache()

    # training report
    lr = optimizer.optimizer.param_groups[0]["lr"]
    mean_loss = np.mean(loss_evol)
    train_duration = str(datetime.timedelta(seconds=int(time.time() - train_time)))
    report = f"{epoch:^5}|{lr:^10.5f}|{mean_loss:^10.3f}|{train_duration:^10}|"

    # Validation -----------------------------------------------------------------------------------------------------
    if epoch % config.validate_every == 0:
        val_time = time.time()
        model.eval()
        validation_results = {}
        with torch.no_grad():
            # If we define validate_n_random_basins as 0 or negative, we take all the basins. Otherwise, we randomly
            # select the number of basins defined in validate_n_random_basins
            if config.validate_n_random_basins <= 0:
                validation_basin_ids = validation_dataset.keys()
            else:
                validation_basin_ids = random.sample(list(validation_dataset.keys()), config.validate_n_random_basins)

            # Go through each basin
            iterator = tqdm(
                validation_basin_ids,
                desc=f"Epoch {epoch}/{config.epochs}. Validation",
                unit="basins",
                ascii=True,
                leave=False,
            )

            for basin in iterator:
                loader = DataLoader(
                    dataset=validation_dataset[basin],
                    batch_size=config.batch_size_evaluation,
                    shuffle=False,
                    drop_last=False,
                    collate_fn=validation_dataset[basin].collate_fn,
                    num_workers=config.num_workers,
                )

                df_ts = pd.DataFrame()
                for sample in loader:
                    sample = upload_to_device(sample, config.device)
                    # Forward pass of the model
                    pred = model(sample)
                    # Backtransform information (unstandardize the output)
                    y_std = validation_dataset[basin].scaler["y_std"].to(config.device)
                    y_mean = validation_dataset[basin].scaler["y_mean"].to(config.device)
                    y_sim = pred["y_hat"] * y_std + y_mean

                    # join results in a dataframe (easier to evaluate/plot later)
                    df = pd.DataFrame(
                        {"y_obs": sample["y_obs"].flatten().cpu().detach(), "y_sim": y_sim.flatten().cpu().detach()},
                        index=pd.to_datetime(sample["date"].flatten()),
                    )

                    df_ts = pd.concat([df_ts, df], axis=0)

                    # remove elements from cuda to free memory
                    del sample, pred, y_sim
                    torch.cuda.empty_cache()

                validation_results[basin] = df_ts

            # average loss validation
            loss_validation = nse(df_results=validation_results)
            report += f"{loss_validation:^10.3f}|{str(datetime.timedelta(seconds=int(time.time() - val_time))):^10}|"

    # No validation
    else:
        report += f"{'':^10}|{'':^10}|"

    # Print report and save model
    config.logger.info(report)
    torch.save(model.state_dict(), config.path_save_folder / "model" / f"model_epoch_{epoch}")
    # modify learning rate
    optimizer.update_optimizer_lr(epoch=epoch)

# print total training time
config.logger.info(f"Total training time: {datetime.timedelta(seconds=int(time.time() - total_time))}\n")

Part 4. Test model

In [ ]:
# In case I already trained an LSTM I can re-construct the model. I just need to define the epoch for which I want to
# re-construct the model
# model = get_model(config).to(config.device)
# model.load_state_dict(torch.load(config.path_save_folder / "model" / "model_epoch_20", map_location=config.device))

In [ ]:
# Read previously generated scaler
with open(config.path_save_folder / "scaler.pickle", "rb") as file:
    scaler = pickle.load(file)

In [ ]:
# In evaluation (validation and testing) we will create an individual dataset per basin
config.logger.info(f"Loading testing data from {config.dataset} dataset")

entities_ids = np.loadtxt(config.path_entities_testing, dtype="str").tolist()
iterator = tqdm(
    [entities_ids] if isinstance(entities_ids, str) else entities_ids,
    desc="Processing entities",
    unit="entity",
    ascii=True,
)

total_time = time.time()
testing_dataset = {}
for entity in iterator:
    dataset = Dataset(cfg=config, time_period="testing", check_NaN=False, entities_ids=entity)

    dataset.scaler = scaler
    dataset.standardize_data(standardize_output=False)
    testing_dataset[entity] = dataset

config.logger.info(
    f"Time required to process {len(iterator)} entities: {datetime.timedelta(seconds=int(time.time() - total_time))}\n"
)

In [ ]:
config.logger.info("Testing model".center(60, "-"))
total_time = time.time()

model.eval()
test_results = {}
with torch.no_grad():
    # Go through each basin
    iterator = tqdm(testing_dataset, desc="Testing", unit="basins", ascii=True)
    for basin in iterator:
        loader = DataLoader(
            dataset=testing_dataset[basin],
            batch_size=config.batch_size_evaluation,
            shuffle=False,
            drop_last=False,
            collate_fn=testing_dataset[basin].collate_fn,
            num_workers=config.num_workers,
        )

        df_ts = pd.DataFrame()
        for sample in loader:
            sample = upload_to_device(sample, config.device)  # upload tensors to device
            pred = model(sample)
            # backtransformed information
            y_sim = pred["y_hat"] * testing_dataset[basin].scaler["y_std"].to(config.device) + (
                testing_dataset[basin].scaler["y_mean"].to(config.device)
            )

            # join results in a dataframe and store them in a dictionary (is easier to plot later)
            df = pd.DataFrame(
                {"y_obs": sample["y_obs"].flatten().cpu().detach(), "y_sim": y_sim.flatten().cpu().detach()},
                index=pd.to_datetime(sample["date"].flatten()),
            )

            df_ts = pd.concat([df_ts, df], axis=0)

            # remove from cuda
            del sample, pred, y_sim
            torch.cuda.empty_cache()

        test_results[basin] = df_ts

# Save results as a pickle file
with open(config.path_save_folder / "test_results.pickle", "wb") as f:
    pickle.dump(test_results, f)

config.logger.info(f"Total testing time: {datetime.timedelta(seconds=int(time.time() - total_time))}")

Part 5. Initial analysis

In [ ]:
# Loss testing
loss_testing = nse(df_results=test_results, average=False)
df_NSE = pd.DataFrame(data={"basin_id": testing_dataset.keys(), "NSE": np.round(loss_testing, 3)})
df_NSE = df_NSE.set_index("basin_id")
df_NSE.to_csv(config.path_save_folder / "NSE_testing.csv", index=True, header=True)

# Plot the histogram
plt.hist(df_NSE["NSE"], bins=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])

# Add NSE statistics to the plot
plt.text(
    0.01,
    0.8,
    (
        f"Mean: {'%.2f' % df_NSE['NSE'].mean():>7}\n"
        f"Median: {'%.2f' % df_NSE['NSE'].median():>0}\n"
        f"Max: {'%.2f' % df_NSE['NSE'].max():>9}\n"
        f"Min: {'%.2f' % df_NSE['NSE'].min():>10}"
    ),
    transform=plt.gca().transAxes,
    bbox=dict(facecolor="white", alpha=0.5),
)

# Format plot
plt.rcParams["figure.figsize"] = (20, 5)
plt.xlabel("NSE", fontsize=12, fontweight="bold")
plt.ylabel("Frequency", fontsize=12, fontweight="bold")
plt.title("NSE Histogram", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Plot simulated and observed discharges
basin_to_analyze = random.sample(list(test_results.keys()), 1)[0]

plt.figure(figsize=(20, 7))
plt.plot(test_results[basin_to_analyze]["y_obs"], label="observed", color=color_palette["observed"])
plt.plot(test_results[basin_to_analyze]["y_sim"], label="simulated", alpha=0.5, color=color_palette["simulated"])

# Format plot
plt.xlabel("Date", fontsize=12, fontweight="bold")
plt.ylabel("Discharge [mm/d]", fontsize=12, fontweight="bold")
plt.title("Modeling results", fontsize=16, fontweight="bold")
plt.tick_params(axis="both", which="major", labelsize=12)
plt.legend(loc="upper right", fontsize=12)
plt.tight_layout()
plt.show()